# LeetCode 442 — Find All Duplicates in an Array bằng Python

Link đề: https://leetcode.com/problems/find-all-duplicates-in-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sorting (và mẹo "dùng chính mảng làm bảng đánh dấu")

Skeleton LeetCode:

```python
class Solution:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code (đối chiếu với cách đếm bằng `Counter`), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng `nums` dài `n`, mọi giá trị nằm trong `[1, n]`, mỗi giá trị xuất hiện **một hoặc hai lần**; cần trả về **các giá trị xuất hiện hai lần**, trong thời gian `O(n)` và chỉ dùng bộ nhớ phụ **hằng số** (không tính danh sách kết quả).
2. Nắm các công cụ Python: chỉ số mảng `nums[v - 1]`, `set`, `collections.Counter`, `abs()`, phép gán song song `a, b = b, a`, `sorted`, `zip(a, a[1:])`, list comprehension với `enumerate`.
3. Đi từ **hai vòng lặp lồng nhau `O(n²)`** → **tập hợp/bộ đếm `O(n)` thời gian, `O(n)` bộ nhớ** → **sắp xếp rồi so hàng xóm** → **đánh dấu bằng dấu âm** → **hoán đổi đưa mỗi giá trị về đúng ô (cycle sort)**; hai cách cuối đạt `O(n)` thời gian, `O(1)` bộ nhớ phụ.
4. Hiểu mấu chốt: vì giá trị nằm trong `[1, n]`, **mỗi giá trị `v` có một "ngôi nhà" riêng là ô `v - 1`** — ta có thể dùng chính mảng làm bảng ghi nhớ mà không cần thêm bộ nhớ.
5. Tránh các lỗi kinh điển: quên `abs()` sau khi đã đổi dấu, đổi dấu hai lần làm mất dấu vết, vòng `while` hoán đổi lặp vô hạn khi có phần tử trùng, và nhầm `v` với `v - 1` khi làm chỉ số.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums` có độ dài `n`, trong đó **mọi số nằm trong khoảng `[1, n]`** và mỗi số xuất hiện **một lần hoặc hai lần**. Hãy trả về một mảng gồm **tất cả các số xuất hiện đúng hai lần**.

Bạn phải viết thuật toán chạy trong `O(n)` thời gian và chỉ dùng **bộ nhớ phụ hằng số** (không tính mảng kết quả).

**Ví dụ 1**

- Input: `nums = [4,3,2,7,8,2,3,1]`
- Output: `[2,3]`
- Giải thích: `2` và `3` mỗi số xuất hiện hai lần, các số còn lại một lần.

**Ví dụ 2**

- Input: `nums = [1,1,2]`
- Output: `[1]`

**Ví dụ 3**

- Input: `nums = [1]`
- Output: `[]`

**Ràng buộc:**

- `n == nums.length`
- `1 <= n <= 10^5`
- `1 <= nums[i] <= n`
- Mỗi phần tử của `nums` xuất hiện **một hoặc hai lần**.

Chú ý: thứ tự các số trong kết quả **không quan trọng** (ví dụ `[3,2]` cũng được chấp nhận); trong notebook ta so sánh sau khi `sort`.

In [1]:
from typing import List
from collections import Counter
import random
import time

examples = [
    ([4, 3, 2, 7, 8, 2, 3, 1], [2, 3]),
    ([1, 1, 2], [1]),
    ([1], []),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected, "| n =", len(nums), "| min/max =", min(nums), max(nums))

[4, 3, 2, 7, 8, 2, 3, 1] -> kỳ vọng [2, 3] | n = 8 | min/max = 1 8
[1, 1, 2] -> kỳ vọng [1] | n = 3 | min/max = 1 2
[1] -> kỳ vọng [] | n = 1 | min/max = 1 1


## 3. Kiến thức nền

### 3.1. Chỉ số của list và "giá trị làm chỉ số"

List trong Python đánh chỉ số từ `0`. Mảng dài `n` có các ô `0, 1, ..., n - 1`. Còn đề cho giá trị trong `[1, n]`, tức là **lệch đúng 1** so với chỉ số. Vì vậy giá trị `v` "ứng với" ô `v - 1`. Đây là phép biến đổi quan trọng nhất của cả bài.

In [2]:
nums = [4, 3, 2, 7, 8, 2, 3, 1]
n = len(nums)
print("n =", n, "-> các chỉ số hợp lệ là 0..", n - 1)
for v in (1, 4, 8):
    print("giá trị", v, "ứng với ô", v - 1, "và nums[v - 1] =", nums[v - 1])
# giá trị 9 sẽ ứng với ô 8: ngoài mảng, nhưng đề đảm bảo không có giá trị 9 khi n = 8

n = 8 -> các chỉ số hợp lệ là 0.. 7
giá trị 1 ứng với ô 0 và nums[v - 1] = 4
giá trị 4 ứng với ô 3 và nums[v - 1] = 7
giá trị 8 ứng với ô 7 và nums[v - 1] = 1


### 3.2. `set` và `Counter`

- `set` là tập hợp: không chứa phần tử trùng; kiểm tra `x in s` rất nhanh (trung bình `O(1)`).
- `Counter(iterable)` đếm số lần mỗi phần tử xuất hiện; tra một khóa chưa có cho `0`.

Hai công cụ này cho cách làm `O(n)` thời gian nhưng tốn `O(n)` bộ nhớ.

In [3]:
seen = set()
for x in [4, 3, 2, 2]:
    print(x, "đã thấy?" , x in seen)
    seen.add(x)
print(seen)

cnt = Counter([4, 3, 2, 7, 8, 2, 3, 1])
print(cnt)
print([v for v, c in cnt.items() if c == 2])
print(cnt[99])     # khóa chưa có -> 0

4 đã thấy? False
3 đã thấy? False
2 đã thấy? False
2 đã thấy? True
{2, 3, 4}
Counter({3: 2, 2: 2, 4: 1, 7: 1, 8: 1, 1: 1})
[3, 2]
0


### 3.3. Phép gán song song và thứ tự vế phải/vế trái

Python cho phép `a, b = b, a`: **toàn bộ vế phải được tính xong trước**, rồi mới gán lần lượt từ trái sang phải. Với chỉ số phụ thuộc lẫn nhau (`nums[nums[i] - 1]`) thứ tự gán cực kỳ quan trọng — ta sẽ gặp lại ở cách cycle sort.

In [4]:
a, b = 1, 2
a, b = b, a
print(a, b)                       # 2 1

nums = [3, 1, 2]
i = 0
# vế phải tính trước: (nums[nums[0]-1], nums[0]) = (nums[2], nums[0]) = (2, 3)
nums[nums[i] - 1], nums[i] = nums[i], nums[nums[i] - 1]
print(nums)                       # [2, 1, 3]

2 1
[2, 1, 3]


### 3.4. Đổi dấu để "đánh dấu đã thăm" và `abs()`

Ta có thể dùng **dấu của số** làm một bit ghi nhớ: số dương = chưa đánh dấu, số âm = đã đánh dấu. Nhưng sau khi đổi dấu, **giá trị gốc nằm ở `abs(x)`**, nên mỗi lần đọc phải bọc `abs()`. Giá trị trong `[1, n]` luôn dương nên không có số `0` gây nhập nhằng giữa "dương" và "âm".

In [5]:
x = 5
x = -x
print(x, abs(x), x < 0)          # -5 5 True
vals = [3, -1, 2]
print([abs(v) for v in vals])    # [3, 1, 2]

-5 5 True
[3, 1, 2]


### 3.5. Sắp xếp và so hàng xóm: `sorted`, `zip(a, a[1:])`

Sau khi sắp xếp, các số bằng nhau đứng cạnh nhau. `zip(a, a[1:])` ghép từng phần tử với phần tử kế tiếp: `(a[0], a[1]), (a[1], a[2]), ...`.

In [6]:
a = sorted([4, 3, 2, 7, 8, 2, 3, 1])
print(a)
print(list(zip(a, a[1:])))
print([x for x, y in zip(a, a[1:]) if x == y])

[1, 2, 2, 3, 3, 4, 7, 8]
[(1, 2), (2, 2), (2, 3), (3, 3), (3, 4), (4, 7), (7, 8)]
[2, 3]


### 3.6. "Bộ nhớ phụ hằng số" nghĩa là gì?

Ta chỉ được phép dùng một số biến đơn lẻ (`i`, `x`, `idx`...) — **không** được tạo thêm list/set/dict có kích thước phụ thuộc vào `n`. Danh sách đáp án `ans` **không bị tính**. Việc sửa trực tiếp `nums` (ghi đè lên chính mảng đầu vào) là hợp lệ trên LeetCode, và chính là chìa khóa của hai cách tối ưu.

### 3.7. List comprehension với `enumerate`

`[v for i, v in enumerate(nums) if điều_kiện]` duyệt đồng thời chỉ số `i` và giá trị `v`, giữ lại `v` thỏa điều kiện — ta sẽ dùng nó ở cách cycle sort.

In [7]:
nums = [1, 2, 3, 2, 3]
print([v for i, v in enumerate(nums) if v != i + 1])   # [2, 3]: ô nào "lạc nhà" thì giữ giá trị của nó

[2, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng một khách sạn có `n` phòng đánh số `1..n` và `n` khách, mỗi khách cầm một **thẻ phòng** ghi số `v` trong `[1, n]`. Có những khách trùng thẻ (hai người cùng cầm thẻ `v`) nên có phòng bị bỏ trống.

Quy tắc: lần lượt cho từng khách vào **đúng phòng ghi trên thẻ**. Nếu phòng `v` đã có người thì khách mới này chính là **thẻ trùng**.

Với `nums = [4,3,2,7,8,2,3,1]` (`n = 8`):

| Khách thứ | Thẻ `v` | Phòng `v` đã có người? | Kết luận |
|---|---|---|---|
| 0 | 4 | chưa | vào phòng 4 |
| 1 | 3 | chưa | vào phòng 3 |
| 2 | 2 | chưa | vào phòng 2 |
| 3 | 7 | chưa | vào phòng 7 |
| 4 | 8 | chưa | vào phòng 8 |
| 5 | 2 | **rồi** | `2` là thẻ trùng |
| 6 | 3 | **rồi** | `3` là thẻ trùng |
| 7 | 1 | chưa | vào phòng 1 |

Đáp án `[2, 3]`. Bảng "phòng nào có người" cần `n` ô nhớ — đó là cách `set`. Bài toán yêu cầu **không dùng thêm ô nhớ**, nên ta phải "mượn" chính mảng `nums` làm bảng ghi nhớ.

In [8]:
nums = [4, 3, 2, 7, 8, 2, 3, 1]
occupied = set()
dups = []
for i, v in enumerate(nums):
    if v in occupied:
        print(f"khách {i}: thẻ {v} -> phòng {v} ĐÃ có người => trùng")
        dups.append(v)
    else:
        print(f"khách {i}: thẻ {v} -> vào phòng {v}")
        occupied.add(v)
print(dups)

khách 0: thẻ 4 -> vào phòng 4
khách 1: thẻ 3 -> vào phòng 3
khách 2: thẻ 2 -> vào phòng 2
khách 3: thẻ 7 -> vào phòng 7
khách 4: thẻ 8 -> vào phòng 8
khách 5: thẻ 2 -> phòng 2 ĐÃ có người => trùng
khách 6: thẻ 3 -> phòng 3 ĐÃ có người => trùng
khách 7: thẻ 1 -> vào phòng 1
[2, 3]


## 5. Cách 1 — Brute Force / baseline: hai vòng lặp so từng cặp

**Ý tưởng.** Với mỗi vị trí `i`, xem có vị trí `j > i` nào mà `nums[j] == nums[i]` hay không. Vì mỗi số xuất hiện tối đa hai lần nên khi thấy cặp trùng, ta thêm `nums[i]` vào kết quả (và mỗi giá trị chỉ được thêm một lần vì chỉ gặp cặp `(i, j)` đúng một lần).

In [9]:
def duplicates_brute(nums):
    n = len(nums)
    ans = []
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] == nums[j]:
                ans.append(nums[i])
                break
    return ans


for nums, expected in examples:
    print(duplicates_brute(nums), "(kỳ vọng", expected, ")")

[3, 2] (kỳ vọng [2, 3] )
[1] (kỳ vọng [1] )
[] (kỳ vọng [] )


**Độ phức tạp (Cách 1).** Hai vòng lặp lồng nhau: **Time `O(n²)`**, **Space `O(1)`** (không tính kết quả). Với `n = 10^5` là khoảng `5 · 10^9` phép so sánh — quá chậm.

**Nút thắt:** để biết "`v` đã xuất hiện trước đó chưa" ta đang **dò lại toàn bộ mảng** mỗi lần. Cần một cách trả lời câu hỏi đó trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Câu hỏi "`v` đã xuất hiện chưa?" thường được trả lời bằng bảng băm (`set`), tốn `O(n)` bộ nhớ. Nhưng đề cho một điều rất đặc biệt: **giá trị nằm trong `[1, n]`** — đúng bằng tập chỉ số `{0..n-1}` dịch đi `1`. Nghĩa là:

> Mảng `nums` có `n` ô; bảng đánh dấu cần `n` ô; **ta có thể dùng chính `nums` làm bảng đánh dấu**.

Có hai cách "gài" thông tin vào mảng mà không làm mất giá trị gốc:

1. **Đánh dấu bằng dấu âm:** gặp giá trị `v`, đổi dấu ô `nums[v - 1]`. Nếu lúc đến ô đó đã âm sẵn thì `v` đã được gặp trước đó → trùng. Giá trị gốc vẫn đọc được qua `abs()`.
2. **Đưa mỗi giá trị về đúng nhà (cycle sort):** hoán đổi để `nums[v - 1] == v`. Cuối cùng ô nào chứa giá trị **không bằng chỉ số + 1** chứa chính số trùng (số "dư", không có chỗ để về).

Ngoài hai cách này, ta cũng xem hai cách dễ hơn để so sánh: dùng `set/Counter` (`O(n)` bộ nhớ) và sắp xếp (`O(n log n)`).

## 7. Các cách tiếp theo

Tổng cộng ta có **5 cách**: (1) brute force ở mục 5; (2) `set`/`Counter`; (3) sắp xếp; (4) đánh dấu bằng dấu âm; (5) cycle sort. Mỗi cách bên dưới đều có: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Tập hợp đã thấy (`set`)

**Giải thích từng đoạn.**

- `seen = set()` lưu các giá trị đã gặp.
- Duyệt từng `x`: nếu `x` đã trong `seen` thì đây là lần thứ hai → thêm vào `ans`; ngược lại thêm `x` vào `seen`.
- Vì mỗi giá trị tối đa hai lần nên không bao giờ thêm một giá trị vào `ans` hai lần.

In [10]:
def duplicates_set(nums):
    seen = set()
    ans = []
    for x in nums:
        if x in seen:
            ans.append(x)
        else:
            seen.add(x)
    return ans


for nums, expected in examples:
    print(duplicates_set(nums), "(kỳ vọng", expected, ")")

[2, 3] (kỳ vọng [2, 3] )
[1] (kỳ vọng [1] )
[] (kỳ vọng [] )


In [11]:
class SolutionSet:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        seen = set()
        ans = []
        for x in nums:
            if x in seen:
                ans.append(x)
            else:
                seen.add(x)
        return ans


print(SolutionSet().findDuplicates([4, 3, 2, 7, 8, 2, 3, 1]))

[2, 3]


**Độ phức tạp (Cách 2).** **Time `O(n)`**, **Space `O(n)`** cho `seen`. Đúng thời gian nhưng **vi phạm** yêu cầu bộ nhớ hằng số — vẫn là baseline tốt để đối chiếu.

### 7.2. Cách 3 — Sắp xếp rồi so hàng xóm

**Giải thích từng đoạn.**

- Sắp xếp một bản sao `a = sorted(nums)`; hai số bằng nhau sẽ đứng cạnh nhau.
- Duyệt `i` từ `1` đến `n - 1`: nếu `a[i] == a[i - 1]` thì `a[i]` là số trùng.
- Vì mỗi số tối đa hai lần nên không thể có ba số liên tiếp bằng nhau; mỗi giá trị trùng được ghi đúng một lần.

In [12]:
def duplicates_sort(nums):
    a = sorted(nums)
    ans = []
    for i in range(1, len(a)):
        if a[i] == a[i - 1]:
            ans.append(a[i])
    return ans


for nums, expected in examples:
    print(duplicates_sort(nums), "(kỳ vọng", expected, ")")

[2, 3] (kỳ vọng [2, 3] )
[1] (kỳ vọng [1] )
[] (kỳ vọng [] )


In [13]:
class SolutionSort:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        a = sorted(nums)
        ans = []
        for i in range(1, len(a)):
            if a[i] == a[i - 1]:
                ans.append(a[i])
        return ans


print(SolutionSort().findDuplicates([1, 1, 2]))

[1]


**Độ phức tạp (Cách 3).** **Time `O(n log n)`** (do sắp xếp), **Space `O(n)`** vì `sorted` tạo bản sao (nếu sắp tại chỗ `nums.sort()` thì phụ `O(1)` đến `O(log n)` tùy cài đặt, nhưng làm đổi thứ tự input). Không đạt `O(n)` thời gian.

### 7.3. Cách 4 — Đánh dấu bằng dấu âm

**Giải thích từng đoạn.**

- Với mỗi `x` trong `nums`, lấy giá trị gốc `v = abs(x)` (vì có thể đã bị đổi dấu), ô nhà của nó là `idx = v - 1`.
- Nếu `nums[idx] < 0`: ô đó đã bị đánh dấu — nghĩa là `v` đã được gặp một lần trước → `v` là số trùng, thêm `v` vào `ans`.
- Ngược lại: đây là lần đầu gặp `v`, đổi dấu `nums[idx] = -nums[idx]` để ghi nhớ.
- (Tùy chọn) cuối cùng trả mọi phần tử về dương để **không phá hỏng input**.

Điểm tinh tế: ô `nums[idx]` bị đổi dấu **không phải vì giá trị của nó**, mà vì **"có ai đó mang thẻ `idx + 1`"**. Giá trị gốc của ô đó vẫn giữ nguyên độ lớn, nên khi duyệt tới nó ta vẫn đọc đúng bằng `abs`.

In [14]:
def duplicates_negate(nums):
    nums = nums[:]                  # làm việc trên bản sao để demo không phá input của ví dụ
    ans = []
    for x in nums:
        v = abs(x)
        idx = v - 1
        if nums[idx] < 0:
            ans.append(v)
        else:
            nums[idx] = -nums[idx]
    return ans


for nums, expected in examples:
    print(duplicates_negate(nums), "(kỳ vọng", expected, ")")

[2, 3] (kỳ vọng [2, 3] )
[1] (kỳ vọng [1] )
[] (kỳ vọng [] )


In [15]:
class SolutionNegate:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        ans = []
        for x in nums:
            v = abs(x)
            if nums[v - 1] < 0:
                ans.append(v)
            else:
                nums[v - 1] = -nums[v - 1]
        for i in range(len(nums)):       # khôi phục input
            nums[i] = abs(nums[i])
        return ans


print(SolutionNegate().findDuplicates([4, 3, 2, 7, 8, 2, 3, 1]))

[2, 3]


**Độ phức tạp (Cách 4).** **Time `O(n)`** (hai lượt duyệt), **Space `O(1)`** phụ (không tính `ans`).

### 7.4. Cách 5 — Cycle sort: đưa mỗi giá trị về đúng nhà

**Giải thích từng đoạn.**

- Với mỗi vị trí `i`, trong khi giá trị đang đứng ở `i` chưa nằm đúng nhà, ta hoán đổi nó về ô `nums[i] - 1`.
- Điều kiện dừng: `nums[i] == nums[nums[i] - 1]`. Nó gộp hai trường hợp: (a) `nums[i]` đã ở đúng nhà (`nums[i] - 1 == i`, hai vế là cùng một ô); (b) nhà của nó đã có **một bản sao khác** đứng sẵn → đây là số trùng và ta dừng để không lặp vô hạn.
- Sau khi mọi vị trí xử lý xong, ô `i` hoặc chứa `i + 1` (đúng nhà) hoặc chứa một số **thừa** (bản sao thứ hai). Vậy các `nums[i] != i + 1` chính là các số trùng.

Cẩn thận thứ tự trong phép gán: `nums[nums[i] - 1], nums[i] = nums[i], nums[nums[i] - 1]`. Vế phải được tính trước rồi mới gán; nhưng gán đầu tiên `nums[nums[i]-1] = ...` dùng `nums[i]` cũ, và gán thứ hai `nums[i] = ...` là sau cùng nên không ảnh hưởng chỉ số phía trước. Cách viết này đúng; nếu đổi vế trái thành `nums[i], nums[nums[i] - 1]` thì **sai** (xem mục 13).

In [16]:
def duplicates_cycle(nums):
    nums = nums[:]
    for i in range(len(nums)):
        while nums[i] != nums[nums[i] - 1]:
            j = nums[i] - 1                        # ô nhà của giá trị đang ở i
            nums[i], nums[j] = nums[j], nums[i]    # dùng j đã chốt trước nên an toàn
    return [v for i, v in enumerate(nums) if v != i + 1]


for nums, expected in examples:
    print(duplicates_cycle(nums), "(kỳ vọng", expected, ")")

[3, 2] (kỳ vọng [2, 3] )
[1] (kỳ vọng [1] )
[] (kỳ vọng [] )


In [17]:
class SolutionCycle:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        for i in range(len(nums)):
            while nums[i] != nums[nums[i] - 1]:
                nums[nums[i] - 1], nums[i] = nums[i], nums[nums[i] - 1]
        return [v for i, v in enumerate(nums) if v != i + 1]


print(SolutionCycle().findDuplicates([4, 3, 2, 7, 8, 2, 3, 1]))

[3, 2]


**Độ phức tạp (Cách 5).** Mỗi phép hoán đổi đưa ít nhất một giá trị về đúng nhà **vĩnh viễn**, và có tối đa `n` giá trị ⇒ tổng số hoán đổi `<= n`. **Time `O(n)`**, **Space `O(1)`** phụ. Cách này **thay đổi thứ tự** `nums` (được phép).

## 8. TRACE từng bước

### 8.1. Bảng trạng thái — đánh dấu bằng dấu âm (`nums = [4,3,2,7,8,2,3,1]`)

| Bước | `x` đọc | `v = abs(x)` | `idx = v-1` | `nums[idx]` trước | Hành động | `ans` |
|---|---|---|---|---|---|---|
| 1 | 4 | 4 | 3 | 7 | đổi dấu → -7 | [] |
| 2 | 3 | 3 | 2 | 2 | đổi dấu → -2 | [] |
| 3 | -2 | 2 | 1 | 3 | đổi dấu → -3 | [] |
| 4 | -7 | 7 | 6 | 3 | đổi dấu → -3 | [] |
| 5 | 8 | 8 | 7 | 1 | đổi dấu → -1 | [] |
| 6 | 2 | 2 | 1 | **-3** (âm) | trùng! | [2] |
| 7 | -3 | 3 | 2 | **-2** (âm) | trùng! | [2, 3] |
| 8 | -1 | 1 | 0 | 4 | đổi dấu → -4 | [2, 3] |

Ở bước 3 và 4, ta đọc được `-2` và `-7` (do bước 2 và bước 1 đã đổi dấu các ô đó) và vẫn lấy đúng `v = 2`, `v = 7` nhờ `abs`. Để ý: ở bước 6 `x = 2` đọc ra là dương vì ô số `5` (chứa `2` thứ hai) chưa bị đổi dấu — ô bị đổi dấu là ô **nhà** `1`, khác ô đang đọc.

In [18]:
def trace_negate(nums):
    nums = nums[:]
    ans = []
    print("bắt đầu:", nums)
    # duyệt theo chỉ số để thấy rõ x đang đọc là nums[i] tại thời điểm đó (có thể đã đổi dấu)
    for i in range(len(nums)):
        x = nums[i]
        v = abs(x)
        idx = v - 1
        before = nums[idx]
        if before < 0:
            ans.append(v)
            action = "ô nhà đã âm => TRÙNG"
        else:
            nums[idx] = -nums[idx]
            action = f"đổi dấu ô {idx}"
        print(f"i={i} x={x:>3} v={v} idx={idx} nums[idx] trước={before:>3} | {action:<24} | nums={nums} ans={ans}")
    return ans


print(trace_negate([4, 3, 2, 7, 8, 2, 3, 1]))

bắt đầu: [4, 3, 2, 7, 8, 2, 3, 1]
i=0 x=  4 v=4 idx=3 nums[idx] trước=  7 | đổi dấu ô 3              | nums=[4, 3, 2, -7, 8, 2, 3, 1] ans=[]
i=1 x=  3 v=3 idx=2 nums[idx] trước=  2 | đổi dấu ô 2              | nums=[4, 3, -2, -7, 8, 2, 3, 1] ans=[]
i=2 x= -2 v=2 idx=1 nums[idx] trước=  3 | đổi dấu ô 1              | nums=[4, -3, -2, -7, 8, 2, 3, 1] ans=[]
i=3 x= -7 v=7 idx=6 nums[idx] trước=  3 | đổi dấu ô 6              | nums=[4, -3, -2, -7, 8, 2, -3, 1] ans=[]
i=4 x=  8 v=8 idx=7 nums[idx] trước=  1 | đổi dấu ô 7              | nums=[4, -3, -2, -7, 8, 2, -3, -1] ans=[]
i=5 x=  2 v=2 idx=1 nums[idx] trước= -3 | ô nhà đã âm => TRÙNG     | nums=[4, -3, -2, -7, 8, 2, -3, -1] ans=[2]
i=6 x= -3 v=3 idx=2 nums[idx] trước= -2 | ô nhà đã âm => TRÙNG     | nums=[4, -3, -2, -7, 8, 2, -3, -1] ans=[2, 3]
i=7 x= -1 v=1 idx=0 nums[idx] trước=  4 | đổi dấu ô 0              | nums=[-4, -3, -2, -7, 8, 2, -3, -1] ans=[2, 3]
[2, 3]


### 8.2. Bảng trạng thái — cycle sort (`nums = [4,3,2,7,8,2,3,1]`)

Mỗi dòng là một phép hoán đổi (hoặc một lần dừng ở vị trí `i`).

| `i` | Hoán đổi | `nums` sau |
|---|---|---|
| 0 | 4 về ô 3 (đổi với 7) | [7,3,2,4,8,2,3,1] |
| 0 | 7 về ô 6 (đổi với 3) | [3,3,2,4,8,2,7,1] |
| 0 | 3 về ô 2: `nums[2]=2 != 3` → đổi | [2,3,3,4,8,2,7,1] |
| 0 | 2 về ô 1: `nums[1]=3 != 2` → đổi | [3,2,3,4,8,2,7,1] |
| 0 | 3 về ô 2: `nums[2]=3 == 3` → **dừng** (trùng) | |
| 1,2,3,4 | ... tiếp tục tương tự | |

Sau khi xử lý hết, mảng thành `[1,2,3,4,3,2,7,8]` (xem output bên dưới): ô 4 chứa `3` và ô 5 chứa `2` đều **lạc nhà** (nhà của chúng, ô 2 và ô 1, đã có bản chính) — đó chính là hai số trùng `3` và `2`.

In [19]:
def trace_cycle(nums):
    nums = nums[:]
    print("bắt đầu:", nums)
    for i in range(len(nums)):
        while nums[i] != nums[nums[i] - 1]:
            j = nums[i] - 1
            print(f"i={i}: giá trị {nums[i]} về ô {j} (đang chứa {nums[j]}) -> đổi")
            nums[i], nums[j] = nums[j], nums[i]
            print("      nums =", nums)
        print(f"i={i}: dừng vì nums[i]={nums[i]} và ô nhà {nums[i] - 1} đã chứa {nums[nums[i] - 1]}")
    ans = [v for i, v in enumerate(nums) if v != i + 1]
    print("ô lạc nhà:", [(i, v) for i, v in enumerate(nums) if v != i + 1])
    return ans


print(trace_cycle([4, 3, 2, 7, 8, 2, 3, 1]))

bắt đầu: [4, 3, 2, 7, 8, 2, 3, 1]
i=0: giá trị 4 về ô 3 (đang chứa 7) -> đổi
      nums = [7, 3, 2, 4, 8, 2, 3, 1]
i=0: giá trị 7 về ô 6 (đang chứa 3) -> đổi
      nums = [3, 3, 2, 4, 8, 2, 7, 1]
i=0: giá trị 3 về ô 2 (đang chứa 2) -> đổi
      nums = [2, 3, 3, 4, 8, 2, 7, 1]
i=0: giá trị 2 về ô 1 (đang chứa 3) -> đổi
      nums = [3, 2, 3, 4, 8, 2, 7, 1]
i=0: dừng vì nums[i]=3 và ô nhà 2 đã chứa 3
i=1: dừng vì nums[i]=2 và ô nhà 1 đã chứa 2
i=2: dừng vì nums[i]=3 và ô nhà 2 đã chứa 3
i=3: dừng vì nums[i]=4 và ô nhà 3 đã chứa 4
i=4: giá trị 8 về ô 7 (đang chứa 1) -> đổi
      nums = [3, 2, 3, 4, 1, 2, 7, 8]
i=4: giá trị 1 về ô 0 (đang chứa 3) -> đổi
      nums = [1, 2, 3, 4, 3, 2, 7, 8]
i=4: dừng vì nums[i]=3 và ô nhà 2 đã chứa 3
i=5: dừng vì nums[i]=2 và ô nhà 1 đã chứa 2
i=6: dừng vì nums[i]=7 và ô nhà 6 đã chứa 7
i=7: dừng vì nums[i]=8 và ô nhà 7 đã chứa 8
ô lạc nhà: [(4, 3), (5, 2)]
[3, 2]


## 9. Code nộp LeetCode

Ta chọn **cách đánh dấu bằng dấu âm** (có khôi phục input) làm lời giải chính: ngắn, `O(n)` thời gian, `O(1)` bộ nhớ phụ và không phá hỏng input. (Cycle sort cũng được chấp nhận.)

```python
class Solution:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        ans = []
        for x in nums:
            v = abs(x)
            if nums[v - 1] < 0:
                ans.append(v)
            else:
                nums[v - 1] = -nums[v - 1]
        for i in range(len(nums)):
            nums[i] = abs(nums[i])
        return ans
```

In [20]:
class Solution:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        ans = []
        for x in nums:
            v = abs(x)
            if nums[v - 1] < 0:
                ans.append(v)
            else:
                nums[v - 1] = -nums[v - 1]
        for i in range(len(nums)):
            nums[i] = abs(nums[i])
        return ans


for nums, expected in examples:
    arr = nums[:]
    got = Solution().findDuplicates(arr)
    print(got, "| input sau khi gọi:", arr, "(đã khôi phục)")
    assert sorted(got) == expected

[2, 3] | input sau khi gọi: [4, 3, 2, 7, 8, 2, 3, 1] (đã khôi phục)
[1] | input sau khi gọi: [1, 1, 2] (đã khôi phục)
[] | input sau khi gọi: [1] (đã khôi phục)


## 10. Vì sao đúng

**Cách dấu âm.** Với mỗi giá trị `v`, ta tác động lên đúng một ô là ô `v - 1`; mỗi ô ứng với đúng một giá trị. Gọi `mark[v]` = "dấu của ô `v - 1`". Ban đầu mọi ô dương. Mỗi lần gặp một **bản sao** của `v`, ta đổi dấu ô `v - 1` **nếu** nó đang dương, và ghi nhận trùng nếu nó đang âm. Vì mỗi giá trị xuất hiện tối đa hai lần:

- lần gặp đầu tiên: ô dương → đổi thành âm;
- lần gặp thứ hai: ô âm → ghi nhận `v` là trùng.

Giá trị chỉ bị đổi dấu bởi các lần gặp **chính nó**, không phải bởi giá trị khác, nên "dấu âm" thực sự có nghĩa "đã gặp `v` một lần". Giá trị độ lớn `|nums[k]|` không đổi nên `abs` luôn trả về giá trị gốc đang đứng ở ô `k` (khi duyệt tới `k`, ta đọc `abs(nums[k])`).

**Cách cycle sort.** Bất biến: sau khi xử lý xong vị trí `i`, ô `i` hoặc chứa `i + 1` hoặc chứa một số có bản sao đã ở đúng nhà của nó. Mỗi phép hoán đổi đặt một giá trị vào đúng nhà (và nó không bao giờ bị lấy ra, vì điều kiện dừng chặn việc hoán đổi khi nhà đã đúng). Khi kết thúc, mỗi giá trị xuất hiện **ít nhất một lần** có một bản ở đúng nhà; một giá trị xuất hiện hai lần có bản thứ hai nằm ở một ô "lạc nhà", và đó là các ô `nums[i] != i + 1`. Ô lạc nhà chứa số trùng nên kết quả đúng.

**Dừng.** Vòng `while` mỗi lần hoán đổi tăng số ô "đúng nhà" thêm ít nhất `1`, tối đa `n` lần; nên không lặp vô hạn.

## 11. Time / Space Complexity

- **Cách dấu âm (lời giải chính):** Time `O(n)`, Space `O(1)` phụ.
- **Cycle sort:** Time `O(n)`, Space `O(1)` phụ.
- **`set`/`Counter`:** Time `O(n)`, Space `O(n)`.
- **Sắp xếp:** Time `O(n log n)`, Space `O(n)` (hoặc `O(1)` nếu sort tại chỗ).
- **Brute force:** Time `O(n²)`, Space `O(1)`.

Đề cho `n <= 10^5`, nên `O(n²)` (khoảng `5 · 10^9`) không thể chạy kịp; ba cách còn lại đều qua giới hạn thời gian, chỉ hai cách cuối thỏa yêu cầu bộ nhớ hằng số.

## 12. Bộ test

Ta gom các cách vào `impls`. Mỗi lần gọi đều truyền **bản sao** của `nums` (vì vài cách sửa tại chỗ). Kết quả có thể ở thứ tự bất kỳ nên so sánh sau khi `sorted`. Oracle độc lập: dùng `Counter` và lấy các giá trị có đếm bằng `2`.

In [21]:
def oracle(nums):
    return sorted(v for v, c in Counter(nums).items() if c == 2)


impls = {
    "brute": duplicates_brute,
    "set": duplicates_set,
    "sort": duplicates_sort,
    "negate": duplicates_negate,
    "cycle": duplicates_cycle,
    "leetcode": lambda a: Solution().findDuplicates(a),
    "submit_set": lambda a: SolutionSet().findDuplicates(a),
    "submit_sort": lambda a: SolutionSort().findDuplicates(a),
    "submit_negate": lambda a: SolutionNegate().findDuplicates(a),
    "submit_cycle": lambda a: SolutionCycle().findDuplicates(a),
}


def check_all(nums, expected, label):
    assert oracle(nums) == expected, (label, "oracle")
    for key, fn in impls.items():
        actual = fn(nums[:])
        assert sorted(actual) == expected, (label, key, nums, actual, expected)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [22]:
check_all([1], [], "n = 1")
check_all([1, 1], [1], "n = 2 trùng")
check_all([1, 2], [], "n = 2 không trùng")
check_all([2, 2], [2], "n = 2, giá trị 2 hai lần (thiếu 1)")
check_all([1, 2, 3, 4, 5], [], "hoán vị đã sắp, không trùng")
check_all([5, 4, 3, 2, 1], [], "hoán vị ngược, không trùng")
check_all([2, 1, 2, 1], [1, 2], "hai giá trị trùng, thiếu 3 và 4")
check_all([1, 1, 3, 3, 5, 5], [1, 3, 5], "mọi giá trị lẻ trùng")
check_all([3, 3, 1, 1, 2, 2][:6], [1, 2, 3], "n = 6 nhưng giá trị 4..6 vắng (hợp lệ: mỗi số tối đa 2 lần)")
check_all([4, 3, 2, 7, 8, 2, 3, 1], [2, 3], "ví dụ đề (lặp lại để chắc)")

PASS n = 1
PASS n = 2 trùng
PASS n = 2 không trùng
PASS n = 2, giá trị 2 hai lần (thiếu 1)
PASS hoán vị đã sắp, không trùng
PASS hoán vị ngược, không trùng
PASS hai giá trị trùng, thiếu 3 và 4
PASS mọi giá trị lẻ trùng
PASS n = 6 nhưng giá trị 4..6 vắng (hợp lệ: mỗi số tối đa 2 lần)
PASS ví dụ đề (lặp lại để chắc)


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi hay gặp: (a) đọc `nums[i]` mà **quên `abs`** sau khi ô đã bị đổi dấu; (b) **nhìn nhầm ô**: kiểm tra dấu của chính `x` thay vì dấu của ô nhà `nums[v - 1]`; (c) dùng `v` thay vì `v - 1` làm chỉ số (lệch một ô).

In [23]:
def wrong_no_abs(nums):
    nums = nums[:]
    ans = []
    for x in nums:
        idx = x - 1            # SAI: x có thể đã âm -> idx âm -> chỉ vào ô cuối mảng một cách im lặng
        if nums[idx] < 0:
            ans.append(abs(x))
        else:
            nums[idx] = -nums[idx]
    return ans


def wrong_check_self(nums):
    nums = nums[:]
    ans = []
    for x in nums:
        v = abs(x)
        if x < 0:                  # SAI: nhìn dấu của chính x thay vì dấu của ô nhà nums[v - 1]
            ans.append(v)
        else:
            nums[v - 1] = -nums[v - 1]
    return ans


def wrong_off_by_one(nums):
    nums = nums[:]
    ans = []
    for x in nums:
        v = abs(x)
        if v < len(nums) and nums[v] < 0:     # SAI: dùng v thay vì v - 1
            ans.append(v)
        elif v < len(nums):
            nums[v] = -nums[v]
    return ans


sample = [4, 3, 2, 7, 8, 2, 3, 1]
bad_abs_input = [3, 3, 1, 1, 2, 2]
print("không abs      :", wrong_no_abs(bad_abs_input), "| đúng:", oracle(bad_abs_input))
print("nhìn nhầm ô    :", wrong_check_self(sample), "| đúng:", oracle(sample))
bad_off_input = [3, 3, 1]
print("lệch một ô     :", wrong_off_by_one(bad_off_input), "| đúng:", oracle(bad_off_input), "(lỗi khi giá trị n bị trùng: không có ô số n)")
assert sorted(wrong_no_abs(bad_abs_input)) != oracle(bad_abs_input)
assert sorted(wrong_check_self(sample)) != oracle(sample)
assert sorted(wrong_off_by_one(bad_off_input)) != oracle(bad_off_input)

không abs      : [3] | đúng: [1, 2, 3]
nhìn nhầm ô    : [2, 7, 1] | đúng: [2, 3]
lệch một ô     : [] | đúng: [3] (lỗi khi giá trị n bị trùng: không có ô số n)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `nums` đúng ràng buộc: bắt đầu từ hoán vị `1..n`, chọn `d` giá trị để nhân đôi và `d` giá trị khác để bỏ đi (nên tổng số phần tử vẫn là `n`, mỗi giá trị xuất hiện 0, 1 hoặc 2 lần; ràng buộc đề nói mỗi phần tử có mặt xuất hiện 1 hoặc 2 lần — thỏa). Sau đó xáo trộn. So mọi cách với oracle `Counter`.

In [24]:
def gen(rng, n):
    values = list(range(1, n + 1))
    rng.shuffle(values)
    d = rng.randint(0, n // 2)
    doubled = values[:d]
    removed = values[d:2 * d]
    removed_set = set(removed)
    arr = [v for v in range(1, n + 1) if v not in removed_set] + doubled
    rng.shuffle(arr)
    assert len(arr) == n and all(1 <= x <= n for x in arr) and max(Counter(arr).values()) <= 2
    return arr


rng = random.Random(442)
total_dups = 0
for trial in range(3000):
    n = rng.randint(1, 12)
    nums = gen(rng, n)
    expected = oracle(nums)
    total_dups += len(expected)
    for key, fn in impls.items():
        assert sorted(fn(nums[:])) == expected, (key, nums)
print("3000 test nhỏ OK; tổng số giá trị trùng đã gặp:", total_dups)

for trial in range(300):
    n = rng.randint(100, 400)
    nums = gen(rng, n)
    expected = oracle(nums)
    for key, fn in impls.items():
        assert sorted(fn(nums[:])) == expected, (key, "vừa")
print("300 test cỡ vừa OK")

3000 test nhỏ OK; tổng số giá trị trùng đã gặp: 4402


300 test cỡ vừa OK


### 12.4. Hiệu năng với `n = 10^5`

Brute force bỏ ra khỏi phép thử này (quá chậm). Ba cách `O(n)`/`O(n log n)` chạy nhanh.

In [25]:
rng = random.Random(4420)
big = gen(rng, 100000)
expected_big = oracle(big)
fast_impls = {k: v for k, v in impls.items() if k != "brute"}
for key, fn in fast_impls.items():
    t0 = time.perf_counter()
    got = fn(big[:])
    dt = time.perf_counter() - t0
    assert sorted(got) == expected_big, key
    print(f"{key:>14}: {len(got)} số trùng, {dt * 1000:.0f} ms ")
print("n = 100000 OK")

           set: 47596 số trùng, 10 ms 
          sort: 47596 số trùng, 25 ms 
        negate: 47596 số trùng, 21 ms 


         cycle: 47596 số trùng, 30 ms 
      leetcode: 47596 số trùng, 27 ms 
    submit_set: 47596 số trùng, 11 ms 
   submit_sort: 47596 số trùng, 24 ms 
 submit_negate: 47596 số trùng, 27 ms 


  submit_cycle: 47596 số trùng, 28 ms 
n = 100000 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `abs()` sau khi đã đổi dấu

**Code sai:**

```python
for x in nums:
    idx = x - 1            # x có thể âm!
```

**Vì sao sai:** nếu `x = -3`, `idx = -4`. Python **không báo lỗi** với chỉ số âm (nó đếm từ cuối), nên ta đọc/ghi nhầm ô và ra đáp án sai một cách im lặng.

**Code đúng:** `v = abs(x); idx = v - 1`.

In [26]:
nums = [3, 3, 1, 1, 2, 2]
bad = wrong_no_abs(nums)
good = Solution().findDuplicates(nums[:])
print("sai:", bad, "| đúng:", sorted(good))
x = -3
print("chỉ số âm vẫn hợp lệ trong list:", "x - 1 =", x - 1, "-> nums[-4] =", nums[x - 1], "(một ô nào đó, không báo lỗi)")

sai: [3] | đúng: [1, 2, 3]
chỉ số âm vẫn hợp lệ trong list: x - 1 = -4 -> nums[-4] = 1 (một ô nào đó, không báo lỗi)


### Lỗi 2: Nhìn nhầm ô — kiểm tra dấu của `x` thay vì dấu của ô nhà

**Code sai:**

```python
for x in nums:
    v = abs(x)
    if x < 0:                 # nhìn dấu của chính x
        ans.append(v)
    else:
        nums[v - 1] = -nums[v - 1]
```

**Vì sao sai:** dấu âm của `x` cho biết "**có ai đó mang thẻ `i + 1`**" (với `i` là vị trí của `x`), chứ **không** cho biết `x` đã xuất hiện trước đó hay chưa. Thông tin "`v` đã xuất hiện" nằm ở **ô nhà của `v`**, tức `nums[v - 1]`.

**Code đúng:** kiểm tra `if nums[v - 1] < 0`.

In [27]:
for t in [[4, 3, 2, 7, 8, 2, 3, 1], [2, 2, 1], [1, 1, 2], [2, 1, 2, 1]]:
    print(t, "nhìn nhầm ô:", sorted(wrong_check_self(t)), "| đúng:", oracle(t))

[4, 3, 2, 7, 8, 2, 3, 1] nhìn nhầm ô: [1, 2, 7] | đúng: [2, 3]
[2, 2, 1] nhìn nhầm ô: [2] | đúng: [2]
[1, 1, 2] nhìn nhầm ô: [] | đúng: [1]
[2, 1, 2, 1] nhìn nhầm ô: [1] | đúng: [1, 2]


### Lỗi 3: Vòng `while` của cycle sort lặp vô hạn

**Code sai:**

```python
while nums[i] != i + 1:
    nums[nums[i] - 1], nums[i] = nums[i], nums[nums[i] - 1]
```

**Vì sao sai:** với số trùng, ví dụ `nums = [1, 1]`: tại `i = 1`, `nums[1] = 1 != 2`, ta đổi `nums[0]` với `nums[1]` — hai ô đều là `1`, mảng **không đổi**, điều kiện vẫn đúng, vòng lặp chạy mãi. Điều kiện đúng là so với **ô nhà**: `nums[i] != nums[nums[i] - 1]`. Dưới đây ta dùng giới hạn số bước để minh họa thay vì chạy vô hạn thật.

In [28]:
def wrong_cycle_limited(nums, limit=50):
    nums = nums[:]
    steps = 0
    for i in range(len(nums)):
        while nums[i] != i + 1:
            nums[nums[i] - 1], nums[i] = nums[i], nums[nums[i] - 1]
            steps += 1
            if steps >= limit:
                return "vòng lặp không dừng (dừng cưỡng bức sau %d bước)" % limit
    return nums


print(wrong_cycle_limited([1, 1]))
print(wrong_cycle_limited([4, 3, 2, 7, 8, 2, 3, 1]))
print("đúng:", duplicates_cycle([1, 1]))

vòng lặp không dừng (dừng cưỡng bức sau 50 bước)
vòng lặp không dừng (dừng cưỡng bức sau 50 bước)
đúng: [1]


### Lỗi 4: Gán song song sai thứ tự vế trái

**Code sai:**

```python
nums[i], nums[nums[i] - 1] = nums[nums[i] - 1], nums[i]
```

**Vì sao sai:** gán `nums[i] = nums[nums[i] - 1]` xảy ra **trước**; sau đó chỉ số trong `nums[nums[i] - 1]` ở vế trái được tính lại bằng `nums[i]` **mới**, nên giá trị cũ được đặt vào sai ô. Cách an toàn là chốt `j = nums[i] - 1` trước rồi đổi `nums[i], nums[j] = nums[j], nums[i]`.

In [29]:
nums = [3, 1, 2]
i = 0
nums[i], nums[nums[i] - 1] = nums[nums[i] - 1], nums[i]     # SAI
print("viết sai thứ tự:", nums, "(mất giá trị 3! thay vì [2, 1, 3])")

nums = [3, 1, 2]
j = nums[i] - 1
nums[i], nums[j] = nums[j], nums[i]                          # ĐÚNG
print("chốt j trước   :", nums)

viết sai thứ tự: [2, 3, 2] (mất giá trị 3! thay vì [2, 1, 3])
chốt j trước   : [2, 1, 3]


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận dạng:** mảng dài `n`, giá trị nằm trong `[1, n]` (hoặc `[0, n-1]`), đề đòi `O(1)` bộ nhớ ⇒ **dùng giá trị làm chỉ số**, và dùng chính mảng làm bảng đánh dấu (bằng dấu âm hoặc hoán đổi).

Bảng chọn nhanh:

| Muốn | Dùng |
|---|---|
| Biết giá trị đã xuất hiện hay chưa, không giới hạn bộ nhớ | `set` |
| Đếm chính xác số lần | `Counter` |
| `O(1)` bộ nhớ, giữ nguyên input (sau khi khôi phục) | đánh dấu dấu âm + khôi phục |
| `O(1)` bộ nhớ, được phép xáo trộn mảng | cycle sort |
| Chỉ cần đúng, đơn giản | sắp xếp rồi so hàng xóm |

Bài cùng họ: **41 First Missing Positive**, **268 Missing Number**, **287 Find the Duplicate Number**, **448 Find All Numbers Disappeared in an Array**, **645 Set Mismatch**.

## 15. Biến thể

### 15.1. Tìm các số **vắng mặt** (LeetCode 448)

Dùng cùng mẹo dấu âm: đánh dấu mọi giá trị đã gặp; sau đó các ô còn **dương** ứng với giá trị chưa từng xuất hiện.

In [30]:
def find_missing(nums):
    nums = nums[:]
    for x in nums:
        v = abs(x)
        if nums[v - 1] > 0:
            nums[v - 1] = -nums[v - 1]
    return [i + 1 for i, x in enumerate(nums) if x > 0]


assert find_missing([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
assert find_missing([1, 1]) == [2]
assert find_missing([1, 2, 3]) == []
print(find_missing([4, 3, 2, 7, 8, 2, 3, 1]))

[5, 6]


### 15.2. Set Mismatch (LeetCode 645)

Mảng là hoán vị `1..n` nhưng một số bị thay bằng bản sao số khác: trả về `[số bị lặp, số bị thiếu]`. Dùng ngay hai kết quả ở trên: số lặp từ `findDuplicates` (một giá trị) và số thiếu từ `find_missing`.

In [31]:
def set_mismatch(nums):
    dup = Solution().findDuplicates(nums[:])[0]
    miss = find_missing(nums)[0]
    return [dup, miss]


assert set_mismatch([1, 2, 2, 4]) == [2, 3]
assert set_mismatch([1, 1]) == [1, 2]
print(set_mismatch([1, 2, 2, 4]))

[2, 3]


### 15.3. Mỗi số xuất hiện tối đa `k` lần?

Dấu âm chỉ mang được **một bit**. Nếu mỗi số có thể xuất hiện nhiều hơn hai lần và ta muốn **đếm**, có thể cộng thêm `n` vào ô nhà mỗi lần gặp: `nums[v - 1] += n`; cuối cùng `(nums[i] - 1) // n` là số lần của giá trị `i + 1` (trừ `1` trước khi chia để ô chứa giá trị gốc `n` không bị tính nhầm thêm một lần; tương tự, khi đọc `x` ta lấy chỉ số bằng `(x - 1) % n`). Đây là cách "mã hóa nhiều thông tin vào một ô".

In [32]:
def count_by_encoding(nums):
    n = len(nums)
    nums = nums[:]
    for x in nums:
        v = (x - 1) % n          # lấy giá trị gốc (đã có thể cộng nhiều lần n)
        nums[v] += n
    return {i + 1: (nums[i] - 1) // n for i in range(n) if (nums[i] - 1) // n > 0}


print(count_by_encoding([4, 3, 2, 7, 8, 2, 3, 1]))
assert count_by_encoding([4, 3, 2, 7, 8, 2, 3, 1]) == {1: 1, 2: 2, 3: 2, 4: 1, 7: 1, 8: 1}
assert count_by_encoding([1, 1, 1]) == {1: 3}

{1: 1, 2: 2, 3: 2, 4: 1, 7: 1, 8: 1}


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | So từng cặp `(i, j)` | `O(n²)` | `O(1)` | Chỉ để hiểu đề / `n` rất nhỏ |
| 2. `set` / `Counter` | Nhớ giá trị đã thấy | `O(n)` | `O(n)` | Phỏng vấn nhanh, không ràng buộc bộ nhớ |
| 3. Sắp xếp | Số trùng đứng cạnh nhau | `O(n log n)` | `O(n)` (hoặc `O(1)` nếu sort tại chỗ) | Khi dễ nghĩ, không cần tối ưu |
| 4. Dấu âm | Ô `v - 1` làm bit "đã thấy `v`" | `O(n)` | `O(1)` | **Lời giải chính** — chuẩn đề, khôi phục được input |
| 5. Cycle sort | Đưa `v` về ô `v - 1`; ô lạc nhà là số trùng | `O(n)` | `O(1)` | Khi được phép sửa/xáo input; nền tảng cho 41, 448 |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `abs(-6)`, `[v for i, v in enumerate([1, 3, 3]) if v != i + 1]`, `Solution().findDuplicates([2, 2, 1])`, `Solution().findDuplicates([1, 2, 3])`, `sorted(Solution().findDuplicates([3, 3, 1, 1, 2, 2]))`.

**Bài 2 (tính tay).** Với `nums = [3, 1, 3, 2, 2]`, lập bảng quá trình đánh dấu bằng dấu âm (từng bước: `x`, `v`, `idx`, `nums[idx]` trước, hành động) và nêu kết quả cuối.

**Bài 3 (điền TODO).** Hoàn thành `dups_todo(nums)` theo cách đánh dấu bằng dấu âm.

In [33]:
def dups_todo(nums):
    ans = []
    for x in nums:
        # TODO 1: v = giá trị gốc của x (nhớ abs)
        # TODO 2: nếu nums[v - 1] < 0 thì ans.append(v)
        # TODO 3: ngược lại đổi dấu nums[v - 1]
        pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `dups_cycle2(nums)` bằng cycle sort **không dùng list comprehension ở cuối**: sau khi đưa các giá trị về nhà, dùng một vòng `for` thường để gom các `nums[i] != i + 1`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `first_duplicate_value(nums)` trả về **số trùng mà lần xuất hiện thứ hai nằm sớm nhất** trong mảng (tức số `x` mà bản sao thứ hai có chỉ số nhỏ nhất), hoặc `-1` nếu không có. Ví dụ `[2, 1, 3, 1, 2]` → `1`. (b) Viết `duplicates_and_count(nums)` trả về `(số_trùng_sắp_tăng, số_giá_trị_vắng_mặt)` chỉ dùng **một** mảng `nums` làm bảng đánh dấu (không `set`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6`, `[3]` (ô 0 chứa `1` đúng nhà; ô 1 chứa `3 != 2`; ô 2 chứa `3 == 3`), `[1]`, `[]`, `[1, 2, 3]`. Bài 3: `v = abs(x)`.
- **Mức 2 (vừa):** Bài 2: x=3 → ô 2 (giá trị 3) đổi thành -3; x=1 → ô 0 thành -3... (viết lại bảng đầy đủ với mảng đang thay đổi). Bài 4: sau vòng `while`, dùng `for i in range(n): if nums[i] != i + 1: ans.append(nums[i])`.
- **Mức 3 (sâu):** Bài 5(a): duyệt theo thứ tự trái → phải với dấu âm; số trùng **đầu tiên được ghi nhận** chính là số có bản sao thứ hai sớm nhất — hãy trả về ngay khi gặp ô nhà đã âm. 5(b): sau lượt đánh dấu, các ô dương cho giá trị vắng mặt; còn số trùng được ghi lại trong lúc đánh dấu.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert abs(-6) == 6
assert [v for i, v in enumerate([1, 3, 3]) if v != i + 1] == [3]
assert sorted(Solution().findDuplicates([2, 2, 1])) == [2]
assert Solution().findDuplicates([1, 2, 3]) == []
assert sorted(Solution().findDuplicates([3, 3, 1, 1, 2, 2])) == [1, 2, 3]

# Bài 2 (cách tự kiểm bằng code)
assert sorted(Solution().findDuplicates([3, 1, 3, 2, 2])) == [2, 3]


# Bài 3
def dups_todo_ans(nums):
    nums = nums[:]
    ans = []
    for x in nums:
        v = abs(x)
        if nums[v - 1] < 0:
            ans.append(v)
        else:
            nums[v - 1] = -nums[v - 1]
    return ans


for nums, expected in examples:
    assert sorted(dups_todo_ans(nums)) == expected


# Bài 4
def dups_cycle2(nums):
    nums = nums[:]
    n = len(nums)
    for i in range(n):
        while nums[i] != nums[nums[i] - 1]:
            j = nums[i] - 1
            nums[i], nums[j] = nums[j], nums[i]
    ans = []
    for i in range(n):
        if nums[i] != i + 1:
            ans.append(nums[i])
    return ans


for nums, expected in examples:
    assert sorted(dups_cycle2(nums)) == expected


# Bài 5
def first_duplicate_value(nums):
    nums = nums[:]
    for x in nums:
        v = abs(x)
        if nums[v - 1] < 0:
            return v
        nums[v - 1] = -nums[v - 1]
    return -1


def duplicates_and_count(nums):
    nums = nums[:]
    dups = []
    for x in nums:
        v = abs(x)
        if nums[v - 1] < 0:
            dups.append(v)
        else:
            nums[v - 1] = -nums[v - 1]
    missing = sum(1 for x in nums if x > 0)
    return sorted(dups), missing


assert first_duplicate_value([2, 1, 3, 1, 2]) == 1
assert first_duplicate_value([1, 2, 3]) == -1
assert first_duplicate_value([4, 3, 2, 7, 8, 2, 3, 1]) == 2
assert duplicates_and_count([4, 3, 2, 7, 8, 2, 3, 1]) == ([2, 3], 2)
assert duplicates_and_count([1]) == ([], 0)
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (giá trị trong `[1, n]`, dùng mảng làm bảng đánh dấu)

```python
class Solution:
    def findDuplicates(self, nums: List[int]) -> List[int]:
        ans = []
        for x in nums:
            v = abs(x)                    # giá trị gốc (có thể ô này đã bị đổi dấu)
            if nums[v - 1] < 0:           # ô nhà của v đã âm -> v đã gặp trước đó
                ans.append(v)
            else:
                nums[v - 1] = -nums[v - 1]
        for i in range(len(nums)):        # khôi phục input (tùy chọn)
            nums[i] = abs(nums[i])
        return ans
```

**Checklist**

- [ ] Có dùng `abs(x)` mỗi lần đọc giá trị trong lúc mảng đang bị đổi dấu?
- [ ] Chỉ số nhà là `v - 1` (không phải `v`)?
- [ ] Chỉ đổi dấu **khi ô đang dương** (không đổi dấu vô điều kiện)?
- [ ] Nếu dùng cycle sort: điều kiện `while nums[i] != nums[nums[i] - 1]` để không lặp vô hạn trên số trùng?
- [ ] Đã thử `[1]`, `[1,1]`, `[2,2]` (thiếu giá trị 1), hoán vị không trùng và ví dụ đề?
- [ ] So sánh kết quả sau khi `sort` (thứ tự đáp án không quan trọng)? Skeleton đúng `def findDuplicates(self, nums: List[int]) -> List[int]`?